# WAV_L1 three-seed paired decision — no GPU
Run only after seed 123 and seed 2026 confirmation notebooks are complete. This notebook reuses frozen seed42 repository evidence and existing seed-matched D0FT evidence, applies the pre-frozen five-criterion gate, writes the three-seed report, and keeps the locked test closed.


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import importlib, json, os, shutil, subprocess, sys
from pathlib import Path
BRANCH='agent/wav1-mechanism-factorization'; REPO=Path('/content/coffee-bean-detection')
if REPO.exists(): shutil.rmtree(REPO)
subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps','-e',str(REPO)],check=True)
sys.path.insert(0,str(REPO/'src')); importlib.invalidate_caches(); os.chdir(REPO)
print('COMMIT:',subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip())


In [ ]:
from coffee_detector.drive_project import require_project_artifact, resolve_drive_project_root
REQ=(
 'experiments/faruq-v3-acmc-paired-confirmation-v1/val_reports/acmc1_paired_optimization_confirmation.json',
 'experiments/faruq-v3-wav-l1-paired-confirmation-v1/seed123/val_reports/WAV_L1_seed123_result.json',
 'experiments/faruq-v3-wav-l1-paired-confirmation-v1/seed2026/val_reports/WAV_L1_seed2026_result.json',
)
PROJECT=resolve_drive_project_root(required_relative_paths=REQ)
D0REF=require_project_artifact(PROJECT,REQ[0]); R123=require_project_artifact(PROJECT,REQ[1]); R2026=require_project_artifact(PROJECT,REQ[2])
SEED42=REPO/'docs/evidence/FARUQ_V3_WAV_L1_SEED42_RESULT_2026-08-19.json'
OUT=PROJECT/'experiments/faruq-v3-wav-l1-paired-confirmation-v1'/'val_reports'/'wav_l1_paired_confirmation.json'
OUT.parent.mkdir(parents=True,exist_ok=True)
print('SEED42:',SEED42); print('SEED123:',R123); print('SEED2026:',R2026); print('D0FT REF:',D0REF)


In [ ]:
CMD=[sys.executable,'-u','-m','coffee_detector.experiments.run_faruq_v3_wav_l1_paired_decision',
 '--seed42-evidence',str(SEED42),'--seed123-result',str(R123),'--seed2026-result',str(R2026),
 '--d0ft-reference',str(D0REF),'--output',str(OUT)]
subprocess.run(CMD,cwd=REPO,check=True)
result=json.loads(OUT.read_text(encoding='utf-8'))
print('=== WAV_L1 THREE-SEED DECISION ===')
for seed,row in result['per_seed'].items():
    print('seed',seed,'D0FT=',row['D0FT'],'WAV_L1=',row['WAV_L1'])
print('\nAGGREGATE:')
for metric,row in result['aggregate'].items(): print(metric,row)
print('\nCRITERIA:',result['criteria'])
print('DECISION:',result['decision'])
print('NEXT:',result['next_action'])
print('REPORT:',OUT)
print('STOP. Kirim output ini untuk review; locked test tetap tertutup.')
